# RTFM scores vs segment index (high-quality, multi-page)

For each training video:
- **x-axis:** segment $1, \ldots, T$  
- **y-axis:** RTFM score  
- **Green band:** human anomaly span (snippet indices)  
- **Red dashed:** $\tau_s$

**Exports (run the plotting cells):**
- **`rtfm_train_segment_scores.pdf`** — one **full page per video** (vector PDF, good for printing / zoom).
- **`pages_png/`** — optional **300 DPI** PNG per video (large bitmaps).
- **`overview_grid.png`** — small multi-panel preview only (lower DPI).

Adjust `PAGE_W_IN`, `PAGE_H_IN`, `PNG_DPI`, and `MAX_VIDEOS_PER_PDF` in the **first code cell** (setup + helpers) if you want bigger pages or **split PDFs**.

### Pooling vs start/end

Start/end define where the caption applies; RTFM scores still **vary** inside that range; the VLM still needs **$K \ll T$** frames → pooling can still help.

**Run all cells.** Colab: fix `REPO_ROOT` if needed.

In [ ]:
# !pip install -q torch matplotlib opencv-python tqdm numpy pandas

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import sys
from pathlib import Path

import cv2
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm

_here = Path.cwd().resolve()
if (_here / "pipeline" / "run_rtfm_pipeline.py").is_file():
    REPO_ROOT = _here
elif (_here.parent / "pipeline" / "run_rtfm_pipeline.py").is_file():
    REPO_ROOT = _here.parent
else:
    REPO_ROOT = Path("/content/explainable-video-anomaly-detection")

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "True")
os.chdir(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "pipeline"))
sys.path.insert(0, str(REPO_ROOT / "rtfm"))

from run_rtfm_pipeline import load_rtfm_model, score_video

ANNOTATIONS = (
    REPO_ROOT
    / "data/SHANGHAI/SHANGHAI_TRAIN/videos_anomalous_train_with_human_annotations/Anomalous_train_annotations.json"
)
TRAIN_LIST = REPO_ROOT / "rtfm/list/shanghai-i3d-train-10crop.list"
FEATURES_ROOT = REPO_ROOT / "rtfm/data/SH_Train_ten_crop_i3d"
VIDEOS_DIR = (
    REPO_ROOT
    / "data/SHANGHAI/SHANGHAI_TRAIN/videos_anomalous_train_with_human_annotations"
)
OUT_DIR = REPO_ROOT / "rtfm_train_viz/outputs/segment_curves"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TAU_S = 0.3

# --- High-quality page export ---
PAGE_W_IN = 14.0   # width (inches) per video page
PAGE_H_IN = 5.0    # height (inches)
PNG_DPI = 300      # per-video PNG raster resolution
SAVE_PNG_PAGES = True
SAVE_MULTI_PAGE_PDF = True
# Split into several PDFs if you hit viewer limits (0 = single file)
MAX_VIDEOS_PER_PDF = 0

# Small notebook preview grid only
GRID_NCOLS = 6
OVERVIEW_GRID_DPI = 120


def resolve_feature_path(line: str, features_root: Path) -> Path | None:
    line = line.strip()
    if not line:
        return None
    p = Path(line)
    if p.is_file():
        return p
    cand = features_root / p.name
    return cand if cand.is_file() else None


def load_train_feature_index(train_list: Path, features_root: Path) -> dict[str, Path]:
    out: dict[str, Path] = {}
    with open(train_list, encoding="utf-8") as f:
        for line in f:
            p = resolve_feature_path(line, features_root)
            if p is None:
                continue
            vid = p.name.replace("_i3d.npy", "").replace(".npy", "")
            out[vid] = p
    return out


def video_frame_count(mp4: Path) -> int:
    cap = cv2.VideoCapture(str(mp4))
    if not cap.isOpened():
        return 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return max(0, n)


def human_span_snippet_bounds(
    anomaly_start: int, anomaly_end: int, total_frames: int, n_snippets: int
) -> tuple[int, int]:
    if total_frames <= 0:
        return 0, max(0, n_snippets - 1)
    fps = total_frames / n_snippets
    s0 = int(anomaly_start / fps)
    s1 = int(anomaly_end / fps)
    s0 = max(0, min(s0, n_snippets - 1))
    s1 = max(0, min(s1, n_snippets - 1))
    if s1 < s0:
        s0, s1 = s1, s0
    return s0, s1


def apply_hq_style():
    mpl.rcParams.update(
        {
            "font.size": 12,
            "axes.titlesize": 15,
            "axes.labelsize": 13,
            "xtick.labelsize": 11,
            "ytick.labelsize": 11,
            "legend.fontsize": 11,
            "axes.linewidth": 1.1,
            "lines.linewidth": 2.0,
            "lines.markersize": 7,
            "figure.dpi": 150,
            "savefig.dpi": PNG_DPI,
        }
    )


def plot_one_video_figure(
    r: dict, tau_s: float, figsize: tuple[float, float]
) -> plt.Figure:
    s = r["scores"]
    T = len(s)
    x = np.arange(1, T + 1)
    s0, s1 = r["human_snip_lo"], r["human_snip_hi"]

    fig, ax = plt.subplots(figsize=figsize, layout="constrained")
    ax.axvspan(s0 + 1 - 0.45, s1 + 1 + 0.45, color="#2d6a4f", alpha=0.2, zorder=0, label="Human span")
    ax.plot(x, s, color="#1b263b", lw=2.2, marker="o", ms=7, zorder=2, label="RTFM score")
    ax.axhline(tau_s, color="#c1121f", ls="--", lw=1.8, zorder=1, label=f"$\\tau_s$ = {tau_s}")
    ax.set_xlim(0.5, T + 0.5)
    if T <= 24:
        ax.set_xticks(np.arange(1, T + 1))
    else:
        step = max(1, T // 16)
        ticks = list(range(1, T + 1, step))
        if ticks[-1] != T:
            ticks.append(T)
        ax.set_xticks(ticks)
    ax.set_xlabel("Segment index")
    ax.set_ylabel("RTFM score (mean logit / 10 crops)")
    ax.set_title(f"{r['video_id']}   —   gate $s_{{\\mathrm{{abn}}}}$ = {r['gate']:.4f}")
    ax.grid(True, alpha=0.35)
    ax.legend(loc="upper right", framealpha=0.95)
    return fig

In [ ]:
with open(ANNOTATIONS, encoding="utf-8") as f:
    annotations = json.load(f)

feat_index = load_train_feature_index(TRAIN_LIST, FEATURES_ROOT)
net, device = load_rtfm_model()

records: list[dict] = []
skipped: list[str] = []

for row in tqdm(annotations, desc="RTFM forward"):
    video_id = row["video_id"]
    if not (row.get("explanation") or "").strip():
        skipped.append(f"{video_id}: empty explanation")
        continue
    fp = feat_index.get(video_id)
    if fp is None:
        skipped.append(f"{video_id}: missing features")
        continue
    mp4 = VIDEOS_DIR / f"{video_id}.mp4"
    if not mp4.is_file():
        skipped.append(f"{video_id}: missing mp4")
        continue
    F = video_frame_count(mp4)
    if F <= 0:
        skipped.append(f"{video_id}: bad frame count")
        continue

    a0 = int(row["anomaly_start_frame"])
    a1 = int(row["anomaly_end_frame"])

    gate, seg_scores = score_video(net, device, str(fp))
    s = np.asarray(seg_scores, dtype=np.float64)
    T = len(s)
    s0, s1 = human_span_snippet_bounds(a0, a1, F, T)

    records.append(
        {
            "video_id": video_id,
            "gate": float(gate),
            "scores": s,
            "human_snip_lo": s0,
            "human_snip_hi": s1,
        }
    )

print(f"Loaded {len(records)} videos; skipped {len(skipped)}")
if skipped:
    print("\n".join(skipped[:15]))
(OUT_DIR / "skipped.txt").write_text("\n".join(skipped) if skipped else "(none)\n", encoding="utf-8")

records_sorted = sorted(records, key=lambda x: x["video_id"])

In [ ]:
summary_path = OUT_DIR / "snippet_score_summary.csv"
with open(summary_path, "w", newline="", encoding="utf-8") as cf:
    w = csv.writer(cf)
    w.writerow(
        [
            "video_id",
            "gate_s_abn",
            "n_segments",
            "human_span_snip_lo",
            "human_span_snip_hi",
            "mean_score",
            "std_score",
            "max_score",
            "argmax_segment_1based",
            f"frac_segments_gt_{TAU_S}",
        ]
    )
    for r in records_sorted:
        s = r["scores"]
        w.writerow(
            [
                r["video_id"],
                f"{r['gate']:.6f}",
                len(s),
                r["human_snip_lo"],
                r["human_snip_hi"],
                f"{float(s.mean()):.6f}",
                f"{float(s.std()):.6f}",
                f"{float(s.max()):.6f}",
                int(s.argmax()) + 1,
                f"{float((s > TAU_S).mean()):.4f}",
            ]
        )
print("Wrote", summary_path)
try:
    import pandas as pd

    display(pd.read_csv(summary_path))
except ImportError:
    pass

In [ ]:
# One large figure per video → multi-page PDF + optional high-DPI PNGs
assert "apply_hq_style" in globals(), "Run the setup cell first (imports + TAU_S + helpers)."
apply_hq_style()
figsize = (PAGE_W_IN, PAGE_H_IN)

png_dir = OUT_DIR / "pages_png"
if SAVE_PNG_PAGES:
    png_dir.mkdir(parents=True, exist_ok=True)

n = len(records_sorted)
chunk = MAX_VIDEOS_PER_PDF if MAX_VIDEOS_PER_PDF > 0 else n
pdf_paths: list[Path] = []

if SAVE_MULTI_PAGE_PDF and n > 0:
    n_parts = max(1, math.ceil(n / chunk))
    for part in range(n_parts):
        lo = part * chunk
        hi = min(n, (part + 1) * chunk)
        batch = records_sorted[lo:hi]
        if n_parts == 1:
            pdf_name = "rtfm_train_segment_scores.pdf"
        else:
            pdf_name = f"rtfm_train_segment_scores_part{part+1:02d}.pdf"
        pdf_path = OUT_DIR / pdf_name
        pdf_paths.append(pdf_path)
        with PdfPages(pdf_path) as pdf:
            for r in tqdm(batch, desc=f"PDF {part+1}/{n_parts}"):
                fig = plot_one_video_figure(r, TAU_S, figsize)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
        print("Wrote", pdf_path, f"({len(batch)} pages)")

if SAVE_PNG_PAGES and n > 0:
    for r in tqdm(records_sorted, desc="PNG pages"):
        fig = plot_one_video_figure(r, TAU_S, figsize)
        p = png_dir / f"{r['video_id']}.png"
        fig.savefig(p, dpi=PNG_DPI, bbox_inches="tight")
        plt.close(fig)
    print("Wrote", len(records_sorted), "PNGs →", png_dir)

mpl.rcParams.update(mpl.rcParamsDefault)

In [ ]:
# Compact multi-panel overview (for quick glance in the notebook / slides)
n = len(records_sorted)
ncols = GRID_NCOLS
nrows = int(np.ceil(n / ncols)) if n else 1
ymin = min(float(r["scores"].min()) for r in records_sorted) if n else 0
ymax = max(float(r["scores"].max()) for r in records_sorted) if n else 1
pad = 0.06 * (ymax - ymin + 1e-9)
ylim = (ymin - pad, ymax + pad)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.0, nrows * 1.5), sharey=True, squeeze=False)
for i, r in enumerate(records_sorted):
    row, col = divmod(i, ncols)
    ax = axes[row][col]
    s = r["scores"]
    T = len(s)
    x = np.arange(1, T + 1)
    s0, s1 = r["human_snip_lo"], r["human_snip_hi"]
    ax.axvspan(s0 + 1 - 0.45, s1 + 1 + 0.45, color="green", alpha=0.12)
    ax.plot(x, s, color="0.2", lw=0.9, marker="o", ms=2)
    ax.axhline(TAU_S, color="crimson", ls="--", lw=0.7)
    ax.set_xlim(0.5, T + 0.5)
    ax.set_ylim(ylim)
    ax.set_title(r["video_id"], fontsize=6)
    ax.tick_params(labelsize=4)
for j in range(n, nrows * ncols):
    row, col = divmod(j, ncols)
    axes[row][col].set_visible(False)
fig.suptitle("Overview (low-res grid) — use PDF or pages_png for full quality", fontsize=10)
fig.tight_layout()
overview_path = OUT_DIR / "overview_grid.png"
fig.savefig(overview_path, dpi=OVERVIEW_GRID_DPI, bbox_inches="tight")
plt.show()
print("Saved", overview_path)

In [ ]:
Ts = {len(r["scores"]) for r in records_sorted}
if len(Ts) != 1:
    print("Varying T across videos:", Ts, "— skip mean±std figure")
else:
    T = Ts.pop()
    mat = np.stack([r["scores"] for r in records_sorted], axis=0)
    mean = mat.mean(axis=0)
    std = mat.std(axis=0)
    x = np.arange(1, T + 1)
    fig2, ax2 = plt.subplots(figsize=(12, 4.5), dpi=120)
    ax2.fill_between(x, mean - std, mean + std, color="steelblue", alpha=0.28)
    ax2.plot(x, mean, color="navy", lw=2.2, marker="o", ms=6)
    ax2.axhline(TAU_S, color="crimson", ls="--", lw=1.5, label=f"$\\tau_s={TAU_S}$")
    ax2.set_xlabel("Segment index")
    ax2.set_ylabel("RTFM score")
    ax2.set_title(f"Across {len(records_sorted)} videos: mean ± std per segment")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0.5, T + 0.5)
    fig2.tight_layout()
    agg_path = OUT_DIR / "rtfm_score_vs_segment_mean_std.png"
    fig2.savefig(agg_path, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved", agg_path)

In [ ]:
s = [x**2 for x in ]